In [1]:
from pathlib import Path
import duckdb

# Locate the project root
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

# Create a persistent DuckDB database
database_file = (
    project_root
    / "data"
    / "processed"
    / "la_restaurant_market.duckdb"
)

connection = duckdb.connect(str(database_file))

print("DuckDB version:", duckdb.__version__)
print("Database created at:")
print(database_file)

DuckDB version: 1.5.5
Database created at:
c:\Users\Scarl\Documents\portfolio\la_chinese_restaurant_site_selection\data\processed\la_restaurant_market.duckdb


In [3]:
# Define the source CSV paths

facility_csv = (
    project_root
    / "data"
    / "interim"
    / "active_restaurant_facilities.csv"
).as_posix()

chinese_csv = (
    project_root
    / "data"
    / "interim"
    / "high_confidence_chinese_candidates.csv"
).as_posix()

# Scan the complete CSV before deciding column types

connection.execute(f"""
    CREATE OR REPLACE TABLE restaurant_facilities AS
    SELECT *
    FROM read_csv_auto(
        '{facility_csv}',
        header = true,
        sample_size = -1
    )
""")

connection.execute(f"""
    CREATE OR REPLACE TABLE high_confidence_chinese AS
    SELECT *
    FROM read_csv_auto(
        '{chinese_csv}',
        header = true,
        sample_size = -1
    )
""")

print("Tables imported successfully.")

Tables imported successfully.


In [4]:
table_check = connection.sql("""
    SELECT
        'restaurant_facilities' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT facility_id) AS unique_facilities
    FROM restaurant_facilities

    UNION ALL

    SELECT
        'high_confidence_chinese' AS table_name,
        COUNT(*) AS row_count,
        COUNT(DISTINCT facility_id) AS unique_facilities
    FROM high_confidence_chinese
""").df()

display(table_check)

,table_name,row_count,unique_facilities
0,restaurant_facilities,27438,27438
1,high_confidence_chinese,559,559


In [5]:
city_competition_sql = connection.sql("""
    WITH all_restaurants AS (
        SELECT
            facility_city,
            COUNT(DISTINCT facility_id) AS all_restaurant_count
        FROM restaurant_facilities
        WHERE special_venue = FALSE
        GROUP BY facility_city
    ),

    chinese_restaurants AS (
        SELECT
            facility_city,
            COUNT(DISTINCT facility_id) AS chinese_candidate_count,
            ROUND(AVG(average_latest_score), 2) AS average_score
        FROM high_confidence_chinese
        GROUP BY facility_city
    )

    SELECT
        a.facility_city,
        a.all_restaurant_count,
        COALESCE(c.chinese_candidate_count, 0)
            AS chinese_candidate_count,
        c.average_score,

        ROUND(
            100.0
            * COALESCE(c.chinese_candidate_count, 0)
            / a.all_restaurant_count,
            2
        ) AS chinese_candidate_share_pct,

        (
            a.all_restaurant_count >= 100
            AND COALESCE(c.chinese_candidate_count, 0) >= 5
        ) AS stable_comparison

    FROM all_restaurants AS a

    LEFT JOIN chinese_restaurants AS c
        ON a.facility_city = c.facility_city

    ORDER BY chinese_candidate_count DESC
""").df()

display(city_competition_sql.head(20))

,facility_city,all_restaurant_count,chinese_candidate_count,average_score,chinese_candidate_share_pct,stable_comparison
0,LOS ANGELES,7265,136,91.90,1.87,True
1,MONTEREY PARK,217,15,90.07,6.91,True
2,TORRANCE,656,13,86.23,1.98,True
3,INDUSTRY,151,13,87.23,8.61,True
4,ALHAMBRA,267,12,88.00,4.49,True
5,POMONA,324,11,92.36,3.40,True
6,ARCADIA,283,11,92.64,3.89,True
7,SAN GABRIEL,247,11,86.27,4.45,True
8,ROSEMEAD,177,10,89.50,5.65,True
9,LANCASTER,366,9,95.78,2.46,True


In [6]:
# Save the SQL result as a reusable database table

connection.register(
    "city_competition_dataframe",
    city_competition_sql
)

connection.execute("""
    CREATE OR REPLACE TABLE preliminary_city_competition AS
    SELECT *
    FROM city_competition_dataframe
""")

print("Derived table created.")

Derived table created.


In [7]:
stable_city_sql = connection.sql("""
    SELECT
        facility_city,
        all_restaurant_count,
        chinese_candidate_count,
        chinese_candidate_share_pct,
        average_score
    FROM preliminary_city_competition
    WHERE stable_comparison = TRUE
    ORDER BY
        chinese_candidate_share_pct DESC,
        chinese_candidate_count DESC
    LIMIT 20
""").df()

display(stable_city_sql)

,facility_city,all_restaurant_count,chinese_candidate_count,chinese_candidate_share_pct,average_score
0,INDUSTRY,151,13,8.61,87.23
1,MONTEREY PARK,217,15,6.91,90.07
2,HACIENDA HEIGHTS,101,6,5.94,93.33
3,ROSEMEAD,177,10,5.65,89.50
4,DIAMOND BAR,133,7,5.26,94.14
5,ALHAMBRA,267,12,4.49,88.00
6,SAN GABRIEL,247,11,4.45,86.27
7,ROWLAND HEIGHTS,220,9,4.09,88.22
8,ARCADIA,283,11,3.89,92.64
9,EL MONTE,250,9,3.60,89.89
